#Transformar Dados de Races

- 1 - Ler a tabela constructors da camada bronze
- 2 - Manter apenas as colunas necessárias para análise (remover a coluna url)
- 3 - Padronizar os nomes das colunas usando snake_case (constructorId → constructor_id)
- 4 - Renomear colunas para deixá-las mais claras (name → constructor_name)
- 5 - Remover registros duplicados
- 6 - Transformar os valores das colunas nationality para Title Case
- 7 - Escrever os dados transformados na tabela constructors da camada silver

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

1 - Ler a tabela constructors da camada bronze

In [0]:
constructors_df = (
    spark.table(bronze_table).filter(((F.col("batch_id")== v_batch_id)))
)

2 - Manter apenas as colunas necessárias para análise (remover a coluna url) usando o método Drop

In [0]:
constructors_dropped_df = constructors_df.drop("url")

3 - Padronizar os nomes das colunas usando snake_case (constructorId → constructor_id) E 4 - Renomear colunas para deixá-las mais claras (name → constructor_name)

In [0]:
constructors_renamed_df = constructors_dropped_df.withColumnsRenamed({
    "constructorId": "constructor_id",
    "name": "constructor_name"
}) 

5 - Remover registros duplicados

In [0]:
constructors_duplicates_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
display(constructors_duplicates_df)

6 - Transformar os valores das colunas nationality para Title Case

In [0]:
constructors_final_df = constructors_duplicates_df.withColumn("nationality", F.initcap(F.col("nationality"))) 

In [0]:
display(constructors_final_df)

constructor_id,constructor_name,nationality,ingestion_timestamp,source_file,batch_id
adams,Adams,American,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01
afm,AFM,German,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01
ags,AGS,French,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01
alfa,Alfa Romeo,Swiss,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01
alphatauri,AlphaTauri,Italian,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01
alpine,Alpine F1 Team,French,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01
alta,Alta,British,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01
amon,Amon,New Zealander,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01
apollon,Apollon,Swiss,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01
arrows,Arrows,British,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01


7 - Escrever os dados transformados na tabela constructors da camada silver

In [0]:
write_to_silver(
    input_df=constructors_final_df,
    target_table=silver_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))

constructor_id,constructor_name,nationality,ingestion_timestamp,source_file,batch_id,created_timestamp,updated_timestamp
ats,ATS,Italian,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
benetton,Benetton,Italian,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
bmw,BMW,German,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
brabham-repco,Brabham-Repco,British,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
cadillac,Cadillac F1 Team,American,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
force_india,Force India,Indian,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
lotus-pw,Lotus-Pratt & Whitney,British,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
osella,Osella,Italian,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
token,Token,British,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
amon,Amon,New Zealander,2026-09-11T22:46:27.875Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/constructors.json,2025-01,2026-09-12T16:01:29.201Z,2026-09-12T16:01:29.201Z
